In [1]:
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time
import psutil
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

print("🚀 Starting CPU-optimized Pneumonia Classification Model")
print("🎯 Using balanced image size: 128x128")

🚀 Starting CPU-optimized Pneumonia Classification Model
🎯 Using balanced image size: 128x128


# معماری مدل:

مدل شامل چندین لایه کانولوشنی با نرمال‌سازی دسته‌ای و افت تصادفی برای جلوگیری از بیش‌برازش است.
   
این مدل با یک لایه چگالی که یک مقدار خروجی (برای طبقه‌بندی باینری) با استفاده از تابع فعال‌سازی سیگموئید دارد، پایان می‌یابد


In [3]:

IMG_SIZE = (128, 128)
BATCH_SIZE = 28

print(f"📐 Image size: {IMG_SIZE}")
print(f"📦 Batch size: {BATCH_SIZE}")

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2,
    fill_mode='constant',
    cval=0
)

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

print("📁 Loading data...")

train_generator = train_datagen.flow_from_directory(
    'chest_xray/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    'chest_xray/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    'chest_xray/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"✅ Data loaded successfully:")
print(f"   🏋️ Training: {train_generator.samples} samples")
print(f"   📊 Validation: {validation_generator.samples} samples")
print(f"   🧪 Test: {test_generator.samples} samples")

📐 Image size: (128, 128)
📦 Batch size: 28
📁 Loading data...
Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 3 classes.
✅ Data loaded successfully:
   🏋️ Training: 4187 samples
   📊 Validation: 1045 samples
   🧪 Test: 624 samples


In [4]:

print("⚖️ Calculating class weights for imbalance...")
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"✅ Class weights: NORMAL={class_weight_dict[0]:.2f}, PNEUMONIA={class_weight_dict[1]:.2f}")


⚖️ Calculating class weights for imbalance...
✅ Class weights: NORMAL=1.94, PNEUMONIA=0.67


# کامپایل مدل Adam 
مدل  بهینه‌ساز و تابع هزینه‌ی آنتروپی باینری که مناسب وظایف طبقه‌بندی باینری است، کامپایل می‌شود


In [6]:


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

print("✅ Model compiled successfully")
model.summary()

✅ Model compiled successfully


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 128, 128, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128, 128, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 128, 128, 32)        │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128, 128, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 64, 64, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64, 64, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 64, 64, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 64, 64, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 64, 64, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 64, 64, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 32, 32, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32, 32, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 32, 32, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 32, 32, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 32, 32, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 32, 32, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 16, 16, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 16, 16, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 4,814,113 (18.36 MB)

 Trainable params: 4,811,937 (18.36 MB)

 Non-trainable params: 2,176 (8.50 KB)

In [5]:
# 4. ساخت مدل بهینه #
# ==================== #

def create_optimized_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
        
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.2),
        
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),
        
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.3),
        
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.4),
        
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    return model

print("🧠 Creating optimized model...")
model = create_optimized_model()


🧠 Creating optimized model...


In [7]:


class SimpleTimeHistory(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []
        self.total_start = time.time()
        print("🔥 Training started!")
    
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()
    
    def on_epoch_end(self, epoch, logs=None):
        epoch_time = time.time() - self.epoch_start
        self.epoch_times.append(epoch_time)
        
        # محاسبه زمان باقی‌مانده
        avg_time = np.mean(self.epoch_times)
        remaining_epochs = self.params['epochs'] - (epoch + 1)
        remaining_time = avg_time * remaining_epochs / 60
        
        # نمایش اطلاعات epoch
        print(f"\n⏰ Epoch {epoch+1}/{self.params['epochs']}")
        print(f"   ⏱️  Time: {epoch_time:.1f}s")
        print(f"   📊 Loss: {logs.get('loss', 'N/A'):.4f} | Acc: {logs.get('accuracy', 'N/A'):.4f}")
        print(f"   📈 Val Loss: {logs.get('val_loss', 'N/A'):.4f} | Val Acc: {logs.get('val_accuracy', 'N/A'):.4f}")
        print(f"   🕐 Est. remaining: {remaining_time:.1f}min")
        print("-" * 50)

# Callback‌های ساده‌تر
callbacks = [
    SimpleTimeHistory(),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=8,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_balanced_model_128.keras',
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]


#  آموزش مدل - با نمایش پیشرفت #

In [ ]:

print("🚀 Starting training with 128x128 images...")
print("📈 Estimated: 3-6 minutes per epoch")

# محاسبه steps مناسب
steps_per_epoch = min(100, len(train_generator))
validation_steps = min(40, len(validation_generator))

print(f"🔢 Steps per epoch: {steps_per_epoch}")
print(f"🔢 Validation steps: {validation_steps}")

start_time = time.time()

# آموزش با verbose=1 برای نمایش پیشرفت
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=validation_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,  # تغییر از 0 به 1 برای نمایش پیشرفت
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps
)

total_time = (time.time() - start_time) / 60
print(f"✅ Training completed in {total_time:.1f} minutes")


🚀 Starting training with 128x128 images...
📈 Estimated: 3-6 minutes per epoch
🔢 Steps per epoch: 100
🔢 Validation steps: 38
🔥 Training started!
Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6593 - auc: 0.7745 - loss: 0.6604 - precision: 0.8870 - recall: 0.6255
⏰ Epoch 1/30
   ⏱️  Time: 340.6s
   📊 Loss: 0.5302 | Acc: 0.7370
   📈 Val Loss: 1.6693 | Val Acc: 0.7426
   🕐 Est. remaining: 164.6min
--------------------------------------------------

Epoch 1: val_auc improved from None to 0.50000, saving model to best_balanced_model_128.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 341s 3s/step - accuracy: 0.7370 - auc: 0.8652 - loss: 0.5302 - precision: 0.9354 - recall: 0.6957 - val_accuracy: 0.7426 - val_auc: 0.5000 - val_loss: 1.6693 - val_precision: 0.7426 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/30
 50/100 ━━━━━━━━━━━━━━━━━━━━ 2:28 3s/step - accuracy: 0.8056 - auc: 0.9365 - loss: 0.3622 - precision: 0.9713 - recall: 0.7599

C:\Users\YOUNES\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



⏰ Epoch 2/30
   ⏱️  Time: 182.0s
   📊 Loss: 0.3802 | Acc: 0.8207
   📈 Val Loss: 2.2234 | Val Acc: 0.7426
   🕐 Est. remaining: 121.9min
--------------------------------------------------

Epoch 2: val_auc did not improve from 0.50000
100/100 ━━━━━━━━━━━━━━━━━━━━ 182s 2s/step - accuracy: 0.8207 - auc: 0.9289 - loss: 0.3802 - precision: 0.9632 - recall: 0.7866 - val_accuracy: 0.7426 - val_auc: 0.5000 - val_loss: 2.2234 - val_precision: 0.7426 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/30
 79/100 ━━━━━━━━━━━━━━━━━━━━ 1:06 3s/step - accuracy: 0.8475 - auc: 0.9436 - loss: 0.3314 - precision: 0.9671 - recall: 0.8246


# ارزیابی
پس از آموزش، مدل بر روی داده‌های تست ارزیابی می‌شود تا دقت آن به دست آید
پیش‌بینی‌هایی بر روی مجموعه تست انجام می‌شود تا گزارشی از طبقه‌بندی و ماتریس سردرگمی تولید شود


In [ ]:
# 8. ارزیابی مدل #
# ==================== #

print("\n📊 Evaluating model on test data...")

test_results = model.evaluate(test_generator, verbose=1)  # تغییر به verbose=1

print("\n" + "="*60)
print("🎯 FINAL TEST RESULTS")
print("="*60)
print(f"📉 Test Loss: {test_results[0]:.4f}")
print(f"🎯 Test Accuracy: {test_results[1]:.4f} ({test_results[1]*100:.2f}%)")
print(f"🎯 Test Precision: {test_results[2]:.4f}")
print(f"🎯 Test Recall: {test_results[3]:.4f}")
print(f"📈 Test AUC: {test_results[4]:.4f}")

# محاسبه F1-Score
precision = test_results[2]
recall = test_results[3]
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"📊 F1-Score: {f1_score:.4f}")

In [ ]:
# پیش‌بینی
print("\n🔍 Making predictions...")
y_pred = model.predict(test_generator, verbose=1)  # تغییر به verbose=1
y_pred_classes = (y_pred > 0.5).astype(int)
y_true = test_generator.classes

print("\n📋 Classification Report:")
print(classification_report(y_true, y_pred_classes, 
                          target_names=['NORMAL', 'PNEUMONIA']))

# تصویربرداری  Seaborn 
ماتریس سردرگمی با استفاده ازبرای تفسیر بهتر نمایش داده می‌شود

In [ ]:
# ماتریس درهم‌ریختگی
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NORMAL', 'PNEUMONIA'],
            yticklabels=['NORMAL', 'PNEUMONIA'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()
